# Eigendecomposition-based IR — equivalence, speed, and gradients

The `_compute_batch_irs` function inside `_tract_ola` runs a Python loop of `L` sequential `_waveguide_step` calls, so its runtime and autograd graph depth both scale linearly with `L`. On GPU this is launch-latency bound — ~8k sequential kernel launches per forward — and an H100 is no faster than a CPU.

`_compute_batch_irs_eig` replaces that loop with an eigendecomposition of the 144×144 state-transition matrix (the waveguide is LTI for a given frame), then evaluates `ir[t] = Σᵢ residueᵢ · λᵢᵗ` as a batched outer product. This notebook checks:

1. **Forward equivalence** — output tensors agree to ~1e-4.
2. **Forward speed** — at the full `L=4096` default used by `pink_trombone_ola`.
3. **Forward+backward speed** — where the eig approach wins most, since the autograd graph has fixed depth (no `L`-deep chain).
4. **Gradient distance** — cosine similarity and relative L2 error between the two gradient tensors.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import time

import numpy as np
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from samuel.pink_trombone import (
    _NOSE_R_CPU,
    _NOSE_START,
    _TRACT_N,
    _compute_batch_irs,
    _compute_batch_irs_eig,
    _compute_diameter_profile,
)

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

## 1. Build realistic reflection coefficients

Reflection coefficients derived from random diameter profiles — same path `_tract_ola` takes. Using random `r` directly produces non-passive systems with `|λ| > 1` that blow up the eig formulation (and the sequential one, over longer L); the diameter→area→r pipeline guarantees a passive, stable `A`.

In [ ]:
BT = 256       # 17 = one ~1.4 s clip worth of frames at 12.5 Hz control rate
N = _TRACT_N
ns = _NOSE_START

g = torch.Generator(device="cpu").manual_seed(0)
tongue_idx = 10 + 20 * torch.rand(BT, 1, generator=g)
tongue_dia = 1.8 + 1.0 * torch.rand(BT, 1, generator=g)
constr_idx = 20 + 20 * torch.rand(BT, 1, generator=g)
constr_dia = 1.5 + 1.5 * torch.rand(BT, 1, generator=g)

diameter = _compute_diameter_profile(tongue_idx, tongue_dia, constr_idx, constr_dia, N).squeeze(1).to(device)
amplitude = diameter ** 2
r_inner = (amplitude[:, :-1] - amplitude[:, 1:]) / (amplitude[:, :-1] + amplitude[:, 1:] + 1e-10)
r = torch.cat([torch.zeros(BT, 1, device=device), r_inner], dim=1)

velum = torch.full((BT,), 0.01, device=device)
A_L = amplitude[:, ns]; A_R = amplitude[:, ns + 1]; A_N = velum ** 2
sA = A_L + A_R + A_N + 1e-10
r_L = (2 * A_L - sA) / sA
r_R = (2 * A_R - sA) / sA
r_N = (2 * A_N - sA) / sA

nose_r = _NOSE_R_CPU.to(device=device, dtype=r.dtype)
inject_pos = torch.zeros(BT, dtype=torch.long, device=device)

print("shapes:", r.shape, r_L.shape, r_R.shape, r_N.shape, nose_r.shape)

## 2. Forward equivalence (L = 4096)

In [ ]:
L = 4096
with torch.no_grad():
    ir_ref = _compute_batch_irs(r, r_L, r_R, r_N, nose_r, True, inject_pos, L)
    ir_eig = _compute_batch_irs_eig(r, r_L, r_R, r_N, nose_r, True, inject_pos, L)

assert ir_ref.shape == ir_eig.shape == (BT, L)
err = (ir_ref - ir_eig).abs()
print(f"shape:            {tuple(ir_ref.shape)}")
print(f"ref max |ir|:     {ir_ref.abs().max().item():.4f}")
print(f"max abs error:    {err.max().item():.2e}")
print(f"RMS error:        {err.pow(2).mean().sqrt().item():.2e}")
print(f"relative RMS err: {err.pow(2).mean().sqrt().item() / ir_ref.pow(2).mean().sqrt().item():.2e}")

In [ ]:
# Overlay a few IRs. They should be visually indistinguishable.
fig = make_subplots(rows=3, cols=1, subplot_titles=[f"Frame {i}" for i in (0, 8, 16)], shared_xaxes=True)
t_axis = np.arange(L) / 44.1  # ms at 44.1 kHz
for row, i in enumerate((0, 8, 16), start=1):
    fig.add_trace(go.Scatter(x=t_axis, y=ir_ref[i].cpu().numpy(), name=f"ref {i}", line=dict(width=1)), row=row, col=1)
    fig.add_trace(go.Scatter(x=t_axis, y=ir_eig[i].cpu().numpy(), name=f"eig {i}", line=dict(width=1, dash="dash")), row=row, col=1)
fig.update_layout(height=600, width=900, template="plotly_white",
                  title="Impulse responses: sequential vs eig",
                  xaxis3_title="time (ms)")
fig.show()

## 3. Forward speed benchmark

In [ ]:
def bench(fn, *args, n_warmup=2, n_trials=5):
    for _ in range(n_warmup):
        fn(*args); sync()
    t0 = time.perf_counter()
    for _ in range(n_trials):
        out = fn(*args)
    sync()
    return (time.perf_counter() - t0) / n_trials, out

Ls = [256, 512, 1024, 2048, 4096]
rows = []
for L in Ls:
    with torch.no_grad():
        t_ref, _ = bench(_compute_batch_irs,     r, r_L, r_R, r_N, nose_r, True, inject_pos, L)
        t_eig, _ = bench(_compute_batch_irs_eig, r, r_L, r_R, r_N, nose_r, True, inject_pos, L)
    rows.append((L, t_ref * 1000, t_eig * 1000, t_ref / t_eig))
    print(f"L={L:5d}  ref={t_ref*1000:7.1f} ms   eig={t_eig*1000:7.1f} ms   speedup={t_ref/t_eig:5.1f}x")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=Ls, y=[row[1] for row in rows], mode="lines+markers", name="sequential"))
fig.add_trace(go.Scatter(x=Ls, y=[row[2] for row in rows], mode="lines+markers", name="eig"))
fig.update_layout(title=f"Forward time (BT={BT}, device={device.type})",
                  xaxis_title="L (IR length)", yaxis_title="ms / call",
                  xaxis_type="log", yaxis_type="log",
                  template="plotly_white", width=700, height=400)
fig.show()

## 4. Forward + backward speed benchmark

This is where the eig approach should win most: the sequential version's backward walks an `L`-deep autograd graph, whereas the eig backward goes through one `eig` call plus O(1) matmul/elementwise ops.

In [ ]:
def bench_fwdbwd(impl, L, n_warmup=1, n_trials=3):
    def run():
        r_ = r.detach().clone().requires_grad_(True)
        ir = impl(r_, r_L, r_R, r_N, nose_r, True, inject_pos, L)
        ir.pow(2).sum().backward()
        return r_.grad
    for _ in range(n_warmup):
        run(); sync()
    t0 = time.perf_counter()
    for _ in range(n_trials):
        run()
    sync()
    return (time.perf_counter() - t0) / n_trials

bwd_rows = []
for L in Ls:
    t_ref = bench_fwdbwd(_compute_batch_irs,     L)
    t_eig = bench_fwdbwd(_compute_batch_irs_eig, L)
    bwd_rows.append((L, t_ref * 1000, t_eig * 1000, t_ref / t_eig))
    print(f"L={L:5d}  ref={t_ref*1000:8.1f} ms   eig={t_eig*1000:8.1f} ms   speedup={t_ref/t_eig:6.1f}x")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=Ls, y=[row[1] for row in bwd_rows], mode="lines+markers", name="sequential"))
fig.add_trace(go.Scatter(x=Ls, y=[row[2] for row in bwd_rows], mode="lines+markers", name="eig"))
fig.update_layout(title=f"Forward+backward time (BT={BT}, device={device.type})",
                  xaxis_title="L (IR length)", yaxis_title="ms / call",
                  xaxis_type="log", yaxis_type="log",
                  template="plotly_white", width=700, height=400)
fig.show()

## 5. Gradient distance

Compute `∂(ir.pow(2).sum())/∂r` under both implementations and report cosine similarity + relative L2 error.

The waveguide geometry routinely produces eigenvalue gaps of ~1e-8, which is at the edge of complex64 precision; `_compute_batch_irs_eig` therefore uses **complex128** internally for the eig path. Complex64 silently returns NaN gradients on the frames where two eigenvalues happen to be that close. complex128 gives ~1e-15 precision, so the 1e-8 gap is huge and `torch.linalg.eig` backward stays clean.

Some drift relative to the sequential reference is still expected (eig backward amplifies perturbations near near-degenerate eigenvalues), but gradients should be finite, tightly aligned (cosine ≈ 1.0), and usable for training.

In [ ]:
def grad_for(impl, L):
    r_ = r.detach().clone().requires_grad_(True)
    ir = impl(r_, r_L, r_R, r_N, nose_r, True, inject_pos, L)
    ir.pow(2).sum().backward()
    return r_.grad.detach()

grad_rows = []
for L in Ls:
    g_ref = grad_for(_compute_batch_irs,     L)
    g_eig = grad_for(_compute_batch_irs_eig, L)
    cos = ((g_ref * g_eig).sum() / (g_ref.norm() * g_eig.norm() + 1e-30)).item()
    rel = ((g_ref - g_eig).norm() / (g_ref.norm() + 1e-30)).item()
    max_abs = (g_ref - g_eig).abs().max().item()
    grad_rows.append((L, cos, rel, max_abs))
    print(f"L={L:5d}  cosine={cos:.6f}  rel L2={rel:.4f}  max |Δ|={max_abs:.3e}")

In [ ]:
# Scatter at the largest L — visual confirmation that gradients track closely.
L = Ls[-1]
g_ref = grad_for(_compute_batch_irs, L).flatten().cpu().numpy()
g_eig = grad_for(_compute_batch_irs_eig, L).flatten().cpu().numpy()
lo = min(g_ref.min(), g_eig.min()); hi = max(g_ref.max(), g_eig.max())
fig = go.Figure()
fig.add_trace(go.Scatter(x=g_ref, y=g_eig, mode="markers",
                         marker=dict(size=4, opacity=0.6), name="∂loss/∂r"))
fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], mode="lines",
                         line=dict(color="gray", dash="dash"), name="y=x"))
fig.update_layout(title=f"Gradient agreement (L={L})",
                  xaxis_title="sequential ∂loss/∂r",
                  yaxis_title="eig ∂loss/∂r",
                  template="plotly_white", width=600, height=500)
fig.show()

## 6. Usage in `_tract_ola`

`_compute_batch_irs_eig` is not wired into `pink_trombone_ola` yet; the public API still uses the sequential `_compute_batch_irs`. If you want to experiment, monkey-patch it locally:

```python
import samuel.pink_trombone as pt
pt._compute_batch_irs = pt._compute_batch_irs_eig
audio = pt.pink_trombone_ola(params, seed=0)
```

Before doing this in training: sanity-check the fit converges using the sequential version first, then swap and compare loss curves. If eig-backward gradient noise bites (visible as loss oscillation late in training, when eigenvalues cluster near strong formants), fall back per-frame or bump to `complex128` inside `_compute_batch_irs_eig`.